# Bloc 1 — Infrastructure de Gestion de Données
## FEM Surrogate ML — RNCP Référentiel

Ce notebook couvre les compétences **C1.1, C1.2, C1.3, C1.4** du Bloc 1.

L'intégralité du pipeline est implémentée avec **Apache Spark** (PySpark) :
lecture distribuée, transformations, chargement dans le Data Warehouse.

| Compétence | Description |
|---|---|
| C1.1 | Architecture Data Lake (MinIO/S3) + Data Warehouse (Parquet structuré) |
| C1.2 | Stockage et calcul distribués via Apache Spark (`local[*]` → cluster YARN) |
| C1.3 | Collecte multi-sources, conformité RGPD (pseudonymisation SHA-256) |
| C1.4 | Pipeline ETL complet : Extract → Transform → Load avec Spark |

## Architecture générale

```
Sources internes          Data Lake          ETL Spark          Data Warehouse
(Simulateur FEniCS)       (MinIO / S3)       (PySpark)          (Parquet structuré)

 sim_v1          ──┐
 sim_v1_wide     ──┤                        ┌─ Extract ─┐
 sim_v1_low      ──┼──▶  bucket             │ Transform │──▶  warehouse.parquet
 sim_v2_moving   ──┤  raw-simulations       └─  Load   ─┘
 ...             ──┘
```

| Couche | Technologie | Rôle |
|---|---|---|
| **Sources** | Simulateur FEniCS (9 variantes) | Production des données brutes |
| **Data Lake** | MinIO (S3-compatible, Docker) | Stockage brut partitionné par date |
| **ETL** | Apache Spark `local[*]` | Calcul distribué multi-cœurs |
| **Data Warehouse** | Parquet structuré | Données propres et pseudonymisées |

In [17]:
import os
import pathlib
import hashlib
import boto3
from botocore.client import Config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

ROOT    = pathlib.Path('..')
RAW_DIR = ROOT / 'data' / 'raw'
WH_DIR  = ROOT / 'data' / 'processed'
WH_PATH = str(WH_DIR / 'warehouse.parquet')

# Fix Windows : HADOOP_HOME + hadoop.dll dans PATH avant démarrage JVM
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin" + os.pathsep + os.environ.get("PATH", "")

spark = (SparkSession.builder
         .appName("FEM-Surrogate-Bloc1")
         .master("local[*]")
         .config("spark.driver.memory", "2g")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"HADOOP_HOME : {os.environ['HADOOP_HOME']}")
print(f"Spark {spark.version} — {spark.sparkContext.defaultParallelism} cœurs disponibles")

HADOOP_HOME : C:\hadoop
Spark 3.5.7 — 20 cœurs disponibles


## 1. Data Lake — Connexion MinIO (C1.1)

MinIO est un object store **compatible S3** déployé via Docker.
Il joue le rôle de **Data Lake** : stockage brut des simulations partitionné par date.

En production, MinIO peut être remplacé par AWS S3 sans modifier le code Spark
(même protocole S3, même configuration `hadoop-aws`).

In [18]:
try:
    s3 = boto3.client(
        's3',
        endpoint_url='http://localhost:9000',
        aws_access_key_id='minioadmin',
        aws_secret_access_key='minioadmin',
        config=Config(signature_version='s3v4'),
        region_name='us-east-1'
    )
    buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
    print("Buckets disponibles dans le Data Lake :")
    for b in buckets:
        print(f"  - {b}")

    # Inventaire des fichiers dans raw-simulations
    response = s3.list_objects_v2(Bucket='raw-simulations')
    objects  = response.get('Contents', [])
    total_mb = sum(o['Size'] for o in objects) / 1e6
    print(f"\nBucket raw-simulations : {len(objects)} fichier(s) — {total_mb:.1f} MB")
    sources = sorted({o['Key'].split('/')[0] for o in objects})
    print("Sources présentes :")
    for src in sources:
        n = sum(1 for o in objects if o['Key'].startswith(src + '/'))
        print(f"  {src:<30} ({n} fichier(s))")

except Exception as e:
    print(f"MinIO non accessible ({e})")
    print("Buckets configurés dans docker-compose :")
    for b in ["raw-simulations", "processed-simulations", "features", "ml-artifacts"]:
        print(f"  - {b}")
    print("\nPour démarrer MinIO : docker compose up -d minio minio-setup")

Buckets disponibles dans le Data Lake :
  - features
  - ml-artifacts
  - processed-simulations
  - raw-simulations

Bucket raw-simulations : 37 fichier(s) — 8.3 MB
Sources présentes :
  sim_v1                         (10 fichier(s))
  sim_v1_low                     (2 fichier(s))
  sim_v1_wide                    (5 fichier(s))
  sim_v1_without_hole            (1 fichier(s))
  sim_v1_without_hole_low        (2 fichier(s))
  sim_v1_without_hole_wide       (5 fichier(s))
  sim_v2_moving_hole             (5 fichier(s))
  sim_v2_moving_hole_low         (2 fichier(s))
  sim_v2_moving_hole_wide        (5 fichier(s))


## 2. Collecte multi-sources (C1.3)

Les données proviennent de **9 sources internes** distinctes (simulateur FEniCS),
chacune représentant une variante géométrique ou de régime de simulation.
Les fichiers sont stockés dans le Data Lake sous forme de **partitions par date**
(équivalent à des exports journaliers de logiciels métier).

Spark lit l'ensemble des sources en une seule instruction et répartit
automatiquement la charge sur les cœurs disponibles.

## 3. Pipeline ETL Spark (C1.2 + C1.4)

Le pipeline suit les trois étapes classiques :

| Étape | Action Spark |
|---|---|
| **Extract** | `spark.read.parquet()` — lecture parallèle multi-fichiers |
| **Transform** | `dropDuplicates`, `dropna`, UDF SHA-256, cast timestamp |
| **Load** | `spark.write.parquet()` — écriture du Data Warehouse |

### 3.1 Extract — Lecture distribuée (C1.2)

Spark découvre et lit **en parallèle** tous les fichiers Parquet des 9 sources.
`mergeSchema` unifie les schémas des différentes variantes (certaines ont des
colonnes supplémentaires comme `hole_x`, `hole_y`).

La colonne `source` est extraite du chemin fichier via `input_file_name()`.

In [19]:
sdf_raw = (spark.read
           .option("mergeSchema", "true")
           .option("recursiveFileLookup", "true")
           .parquet(str(RAW_DIR))
           .withColumn(
               "source",
               F.regexp_extract(F.input_file_name(), r"raw[/\\]([^/\\]+)[/\\]", 1)
           )
           .cache())

print(f"Partitions Spark : {sdf_raw.rdd.getNumPartitions()}")
print(f"Colonnes         : {len(sdf_raw.columns)}")
print()
sdf_raw.printSchema()

print("Inventaire par source :")
(sdf_raw
 .groupBy("source")
 .count()
 .orderBy("count", ascending=False)
 .show(truncate=False))

print(f"Total brut : {sdf_raw.count():,} simulations")

Partitions Spark : 19
Colonnes         : 21

root
 |-- simulation_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- max_displacement_m: double (nullable = true)
 |-- max_von_mises_pa: double (nullable = true)
 |-- solver_name: string (nullable = true)
 |-- solver_version: string (nullable = true)
 |-- data_version: string (nullable = true)
 |-- material_category: string (nullable = true)
 |-- dimension_category: string (nullable = true)
 |-- length_m: double (nullable = true)
 |-- height_m: double (nullable = true)
 |-- young_modulus_pa: double (nullable = true)
 |-- poisson_ratio: double (nullable = true)
 |-- traction_pa: double (nullable = true)
 |-- mesh_nx: long (nullable = true)
 |-- mesh_ny: long (nullable = true)
 |-- hole_radius_ratio: double (nullable = true)
 |-- geometry_type: string (nullable = true)
 |-- hole_cx_ratio: double (nullable = true)
 |-- hole_cy_ratio: double (nullable = true)
 |-- source: string (nullable = false)

Inventaire par sourc

### 3.2 Transform — Nettoyage et pseudonymisation RGPD (C1.3 + C1.4)

Toutes les transformations sont exprimées en **opérations Spark** (lazy evaluation) :
le plan d'exécution est optimisé par le Catalyst Optimizer avant d'être exécuté.

| Étape | API Spark |
|---|---|
| Typage timestamp | `F.to_timestamp()` |
| Suppression doublons | `dropDuplicates(["simulation_id"])` |
| Valeurs manquantes | `dropna(subset=colonnes_critiques)` |
| Pseudonymisation RGPD | UDF `@udf(StringType())` + SHA-256 |

In [20]:
COLONNES_CRITIQUES = [
    "max_displacement_m", "max_von_mises_pa",
    "length_m", "height_m", "young_modulus_pa", "poisson_ratio", "traction_pa"
]

@udf(StringType())
def pseudonymize(uid):
    if uid is None:
        return None
    return hashlib.sha256(uid.encode()).hexdigest()

sdf_clean = (sdf_raw
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .dropDuplicates(["simulation_id"])
    .dropna(subset=COLONNES_CRITIQUES)
    .withColumn("simulation_id", pseudonymize(F.col("simulation_id")))
)

n_before = sdf_raw.count()
n_after  = sdf_clean.count()
print(f"Avant transform : {n_before:,} lignes")
print(f"Après transform : {n_after:,} lignes  ({n_before - n_after} supprimées)")
print()
print("Extrait (simulation_id pseudonymisé) :")
sdf_clean.select("simulation_id", "timestamp", "source", "max_von_mises_pa").show(3, truncate=False)

Avant transform : 61,550 lignes
Après transform : 61,550 lignes  (0 supprimées)

Extrait (simulation_id pseudonymisé) :
+----------------------------------------------------------------+--------------------------+------------------+--------------------+
|simulation_id                                                   |timestamp                 |source            |max_von_mises_pa    |
+----------------------------------------------------------------+--------------------------+------------------+--------------------+
|6f11b4d8953976a6aa078bbcd49c676a3b9691325b4d02f7bf332af2de2782d1|2026-02-17 14:11:32.00038 |sim_v2_moving_hole|3746697.164283065   |
|deb5f164791cfbf565d16f22827d76d9282de1ed70120865b0b843c68dc6d228|2026-02-10 16:25:01.724508|sim_v1            |1430368.3822880955  |
|e6971c01e8438c1a55dc9c8eba379c1171acfa20eda0b13e1f91d6beca060f02|2026-02-17 11:57:14.104397|sim_v2_moving_hole|1.0486613194262397E7|
+----------------------------------------------------------------+----------

### 3.3 Load — Écriture dans le Data Warehouse (C1.4)

Spark écrit le résultat sous forme de fichiers Parquet dans le **bucket MinIO `processed-simulations`**
via le connecteur S3A (`hadoop-aws`). Ce format constitue le **Data Warehouse** du projet :
données propres, typées et pseudonymisées, prêtes pour les analyses des Blocs 2, 3 et 4.

En développement local (sans Docker), l'écriture se fait sur le système de fichiers local.
En production (Docker), le service `spark-etl` écrit directement dans MinIO :

```
# Production — pipeline ETL complet via Docker
docker compose --profile etl run --rm spark-etl
```

In [21]:
WH_DIR.mkdir(parents=True, exist_ok=True)

(sdf_clean
    .write
    .mode("overwrite")
    .parquet(WH_PATH))

n_wh = spark.read.parquet(WH_PATH).count()
print(f"Data Warehouse écrit : {WH_PATH}")
print(f"Lignes chargées      : {n_wh:,}")

Data Warehouse écrit : ..\data\processed\warehouse.parquet
Lignes chargées      : 61,550


## 4. Validation du Data Warehouse (C1.4)

Vérification en trois points avec Spark :
- **Unicité** : aucun doublon sur `simulation_id`
- **Complétude** : zéro valeur manquante sur les colonnes critiques
- **Distribution** : répartition des simulations par source

In [22]:
wh = spark.read.parquet(WH_PATH)

total     = wh.count()
distincts = wh.select("simulation_id").distinct().count()

print(f"=== Dimensions ===")
print(f"{total:,} lignes × {len(wh.columns)} colonnes\n")

print(f"=== Doublons sur simulation_id ===")
print(f"{total - distincts} doublon(s)\n")

print("=== Valeurs manquantes (colonnes critiques) ===")
wh.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in COLONNES_CRITIQUES
]).show()

print("=== Répartition par source ===")
wh.groupBy("source").count().orderBy("count", ascending=False).show(truncate=False)

=== Dimensions ===
61,550 lignes × 21 colonnes

=== Doublons sur simulation_id ===
0 doublon(s)

=== Valeurs manquantes (colonnes critiques) ===
+------------------+----------------+--------+--------+----------------+-------------+-----------+
|max_displacement_m|max_von_mises_pa|length_m|height_m|young_modulus_pa|poisson_ratio|traction_pa|
+------------------+----------------+--------+--------+----------------+-------------+-----------+
|                 0|               0|       0|       0|               0|            0|          0|
+------------------+----------------+--------+--------+----------------+-------------+-----------+

=== Répartition par source ===
+------------------------+-----+
|source                  |count|
+------------------------+-----+
|sim_v1                  |20000|
|sim_v2_moving_hole_wide |10000|
|sim_v1_wide             |10000|
|sim_v1_without_hole_wide|10000|
|sim_v2_moving_hole      |10000|
|sim_v1_without_hole_low |500  |
|sim_v2_moving_hole_low  |500  

### 4.1 Requêtes SQL distribuées (C1.2)

Spark SQL permet d'interroger le Data Warehouse avec la syntaxe SQL standard,
exécutée de façon distribuée sur le cluster — aucune modification de code
lors d'un passage en production sur YARN ou Databricks.

In [23]:
wh.createOrReplaceTempView("warehouse")

# Statistiques agrégées par source
spark.sql("""
    SELECT
        source,
        COUNT(*)                                   AS nb_simulations,
        ROUND(AVG(max_von_mises_pa),    2)         AS von_mises_moyen_pa,
        ROUND(MAX(max_von_mises_pa),    2)         AS von_mises_max_pa,
        ROUND(AVG(max_displacement_m),  8)         AS deplacement_moyen_m
    FROM warehouse
    GROUP BY source
    ORDER BY nb_simulations DESC
""").show(truncate=False)

# Dispersion des contraintes par source
spark.sql("""
    SELECT
        source,
        ROUND(MIN(max_von_mises_pa),    2) AS von_mises_min_pa,
        ROUND(MAX(max_von_mises_pa),    2) AS von_mises_max_pa,
        ROUND(STDDEV(max_von_mises_pa), 2) AS von_mises_ecart_type
    FROM warehouse
    GROUP BY source
    ORDER BY von_mises_ecart_type DESC
""").show(truncate=False)

+------------------------+--------------+------------------+----------------+-------------------+
|source                  |nb_simulations|von_mises_moyen_pa|von_mises_max_pa|deplacement_moyen_m|
+------------------------+--------------+------------------+----------------+-------------------+
|sim_v1                  |20000         |2518405.42        |7818949.72      |1.203E-5           |
|sim_v2_moving_hole_wide |10000         |8566801.85        |4.405073455E8   |6.207E-5           |
|sim_v1_wide             |10000         |4922500.61        |1.706432237E7   |4.669E-5           |
|sim_v1_without_hole_wide|10000         |2489347.56        |9633355.78      |4.235E-5           |
|sim_v2_moving_hole      |10000         |4464957.86        |9.93448746E7    |1.324E-5           |
|sim_v1_without_hole_low |500           |1812107.25        |4373150.43      |1.7222E-4          |
|sim_v2_moving_hole_low  |500           |4424184.47        |2.567748697E7   |2.0396E-4          |
|sim_v1_low         

## 5. Conformité RGPD (C1.3)

Conformément au RGPD, les `simulation_id` ont été **pseudonymisés** par hash
SHA-256 irréversible avant tout chargement dans le Data Warehouse.

- L'identifiant original n'est **jamais stocké** dans le Data Warehouse.
- Le hash SHA-256 (64 caractères hex) est non réversible sans clé externe.
- Aucune donnée à caractère personnel n'est présente dans les simulations FEM.

In [24]:
sample_hash = wh.select("simulation_id").first()[0]
print(f"=== Vérification pseudonymisation ===")
print(f"Longueur des simulation_id : {len(sample_hash)} car. (SHA-256 = 64 hex)\n")

exemple_uuid = "441e59ef-2b2b-4c8b-82ad-23dc69523212"
exemple_hash = hashlib.sha256(exemple_uuid.encode()).hexdigest()
print(f"Exemple :")
print(f"  UUID original : {exemple_uuid}")
print(f"  Hash SHA-256  : {exemple_hash}")

present_uuid = wh.filter(F.col("simulation_id") == exemple_uuid).count()
present_hash = wh.filter(F.col("simulation_id") == exemple_hash).count()
print(f"\nL'UUID original est dans le warehouse : {present_uuid > 0}")
print(f"Le hash SHA-256 est dans le warehouse  : {present_hash > 0}")

=== Vérification pseudonymisation ===
Longueur des simulation_id : 64 car. (SHA-256 = 64 hex)

Exemple :
  UUID original : 441e59ef-2b2b-4c8b-82ad-23dc69523212
  Hash SHA-256  : 77e469a7dd00e3b7f9751110754c1942d5e4e3396e02d3e52bcf3fe73bb6b1bb

L'UUID original est dans le warehouse : False
Le hash SHA-256 est dans le warehouse  : True


In [25]:
spark.stop()
print("Session Spark fermée.")

Session Spark fermée.


## 5. Vérification dans le Data Lake MinIO (C1.1)

Après l'exécution du pipeline `spark-etl` via Docker, le Data Warehouse est disponible
dans le bucket `processed-simulations` de MinIO. On vérifie ici sa présence via boto3.

In [26]:
try:
    processed = s3.list_objects_v2(Bucket='processed-simulations')
    objects   = processed.get('Contents', [])

    if not objects:
        print("Bucket processed-simulations vide.")
        print("Lancer d'abord : docker compose --profile etl run --rm spark-etl")
    else:
        total_mb = sum(o['Size'] for o in objects) / 1e6
        print(f"=== Data Warehouse dans MinIO ===")
        print(f"Bucket   : processed-simulations")
        print(f"Fichiers : {len(objects)}")
        print(f"Taille   : {total_mb:.1f} MB")
        print()
        print("Fichiers présents :")
        for o in objects[:10]:
            print(f"  {o['Key']:<60} {o['Size']/1e3:.0f} KB")
        if len(objects) > 10:
            print(f"  ... ({len(objects) - 10} fichiers supplémentaires)")

except Exception as e:
    print(f"MinIO non accessible : {e}")

=== Data Warehouse dans MinIO ===
Bucket   : processed-simulations
Fichiers : 10
Taille   : 9.0 MB

Fichiers présents :
  warehouse.parquet/_SUCCESS                                   0 KB
  warehouse.parquet/part-00000-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 937 KB
  warehouse.parquet/part-00001-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 941 KB
  warehouse.parquet/part-00002-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 935 KB
  warehouse.parquet/part-00003-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 933 KB
  warehouse.parquet/part-00004-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 930 KB
  warehouse.parquet/part-00005-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 967 KB
  warehouse.parquet/part-00006-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 950 KB
  warehouse.parquet/part-00007-998f900b-3d71-4bb9-addc-5520e52f37c5-c000.snappy.parquet 957 KB
  warehouse.parquet/part-00008-998f900b-3d71-4bb9-ad

## Conclusion — Synthèse du Bloc 1

| Compétence | Réalisé |
|---|---|
| **C1.1** | Architecture Data Lake (MinIO/S3) + Data Warehouse (Parquet structuré) |
| **C1.2** | Apache Spark `local[*]` — lecture parallèle, Catalyst Optimizer, scalable cluster |
| **C1.3** | 9 sources collectées, pseudonymisation RGPD SHA-256 via Spark UDF |
| **C1.4** | Pipeline ETL complet en Spark : `read` → `dropDuplicates/dropna/udf` → `write` |

Le Data Warehouse contient **61 550 simulations** propres, typées et pseudonymisées,
prêtes pour les analyses des Blocs 2, 3 et 4.

---

## Livrable — Étude d'Infrastructure (RNCP Bloc 1)

### Contexte projet

**Projet :** FEM Surrogate ML — modèle de substitution pour simulations éléments finis (FEniCS/DOLFINx)  
**Problématique data :** gérer, nettoyer et rendre accessibles 61 550 simulations physiques issues de 9 variantes géométriques, en respectant les normes RGPD et en garantissant la scalabilité Big Data.

---

### Schéma d'infrastructure

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        SOURCES INTERNES                                 │
│  Simulateur FEniCS/DOLFINx — 9 variantes géométriques                  │
│  sim_v1 · sim_v1_wide · sim_v1_low · sim_v1_without_hole · ...         │
└────────────────────────────┬────────────────────────────────────────────┘
                             │ upload_raw_to_minio.py (boto3)
                             ▼
┌─────────────────────────────────────────────────────────────────────────┐
│              DATA LAKE — MinIO (S3-compatible, Docker)                  │
│  Protocole S3 — remplaçable par AWS S3 sans modification du code        │
│                                                                         │
│  Buckets :                                                              │
│    raw-simulations       → 37 fichiers Parquet, partitionnés par date   │
│    processed-simulations → Data Warehouse transformé                    │
│    features              → features engineered pour ML                  │
│    ml-artifacts          → modèles MLflow                               │
└────────────────────────────┬────────────────────────────────────────────┘
                             │ s3a://raw-simulations/
                             ▼
┌─────────────────────────────────────────────────────────────────────────┐
│              PIPELINE ETL — Apache Spark 3.5 (spark-etl Docker)         │
│                                                                         │
│  EXTRACT   spark.read.parquet("s3a://raw-simulations/")                 │
│            mergeSchema · recursiveFileLookup · 19 partitions · 20 cœurs │
│                                                                         │
│  TRANSFORM dropDuplicates · dropna · to_timestamp · UDF SHA-256        │
│            Catalyst Optimizer · lazy evaluation                         │
│                                                                         │
│  LOAD      spark.write.parquet("s3a://processed-simulations/")          │
│            mode=overwrite · format Parquet columnar                     │
└────────────────────────────┬────────────────────────────────────────────┘
                             │ s3a://processed-simulations/warehouse.parquet
                             ▼
┌─────────────────────────────────────────────────────────────────────────┐
│                DATA WAREHOUSE — MinIO processed-simulations             │
│  61 550 simulations · 21 colonnes · 0 doublon · 0 valeur manquante     │
│  simulation_id pseudonymisé SHA-256 · timestamp UTC · source taguée    │
└─────────────────────────────────────────────────────────────────────────┘
                             │
                             ▼
              ML pipeline · FastAPI · Streamlit · MLflow
```

---

### Choix technologiques

| Besoin | Technologie choisie | Justification |
|---|---|---|
| Data Lake | **MinIO** (Docker) | Open-source, compatible S3, déployable on-premise |
| Calcul distribué | **Apache Spark 3.5** | Scalable vers YARN/Databricks sans modifier le code |
| Connecteur S3 | **hadoop-aws 3.3.4** | Compatible Spark 3.5 / Hadoop 3.3.4 |
| Format de stockage | **Parquet** | Columnar, compressé, lecture rapide, natif Spark |
| ETL | **PySpark DataFrame API** | Lazy evaluation, Catalyst Optimizer, SQL natif |
| Pseudonymisation | **SHA-256** (UDF Spark) | Irréversible, distribuée, conforme RGPD |
| Orchestration | **Docker Compose** + profile `etl` | Job one-shot reproductible |

---

### Scalabilité Big Data

L'architecture est conçue pour monter en charge sans modifier le code :

| Environnement | Spark master | Stockage |
|---|---|---|
| Développement local | `local[*]` | `data/raw/` (filesystem) |
| Docker (actuel) | `local[*]` dans conteneur | MinIO S3 (`s3a://`) |
| Production cluster | `yarn` ou `spark://host:7077` | AWS S3 ou MinIO distribué |

Passage de Docker à YARN : changer une seule ligne — `.master("local[*]")` → `.master("yarn")`

---

### Conformité RGPD

- Les `simulation_id` sont **pseudonymisés** par hash SHA-256 irréversible avant tout stockage dans le Data Warehouse
- L'identifiant original n'est **jamais écrit** dans MinIO processed-simulations
- Aucune donnée à caractère personnel dans les simulations FEM (données purement physiques)
- Principe de **minimisation des données** : seules les colonnes utiles à l'analyse sont conservées